# 01 - AOI & boundaries (Phase 1)

**Colombo UHI practicum.** Builds and visualises every study-area geometry:
Colombo District + Western Province (FAO GAUL), DS/GN divisions (user assets,
filtered to Colombo District), the CMC boundary (~37 km2 sanity check), the
GHSL-derived urban extent, the combined water mask, and BOTH SUHII
rural-reference definitions (`buffer_ring`, `lcz_based`).

Run top-to-bottom in **Google Colab** after `00_setup_and_auth.ipynb` has
worked once. All logic lives in `src/colombo_uhi/`; this notebook only
orchestrates and displays.

> **Caveat (CLAUDE.md #1):** everything in this project is LAND SURFACE
> TEMPERATURE analysis - never air temperature. The exact caveat string is
> printed from `params["caveats"]` below and must accompany every product.

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Which revision is actually on disk. Quote this if a result looks impossible.
!git --no-pager log -1 --format="HEAD %h %s (%ci)"

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00 in this runtime)
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))

# Drop any already-imported colombo_uhi modules BEFORE importing. Without this,
# re-running the notebook in a live runtime keeps the version cached in
# sys.modules from the previous run: `git pull` updates the files on disk but the
# import silently returns the OLD code, so new functions appear to not exist
# (AttributeError) and fixed bugs appear unfixed.
for _name in [m for m in list(sys.modules) if m == "colombo_uhi" or m.startswith("colombo_uhi.")]:
    del sys.modules[_name]

from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
print("CAVEAT:", params["caveats"]["lst_not_air_temp"])

## Step 1 - inspect the uploaded boundary assets

The DS/GN assets are **your uploads**, so their attribute schema depends on the
source: OCHA COD-AB uses `ADM3_EN` / `ADM2_EN`, geoBoundaries uses `shapeName`
and has no parent-district column at all. This cell prints what each asset
actually contains; `aoi._resolve_property()` then picks the right field from the
candidate lists in `params.yaml` (`aoi.assets.*_candidates`).

If a later cell complains that no candidate matched, copy the correct field name
from this output into the matching candidate list.

In [ ]:
# COLAB: RUN THIS CELL
from colombo_uhi import aoi

# Fail fast and legibly if an OLD colombo_uhi is loaded (see the purge in the
# auth cell) or if the repo on disk predates the functions this notebook needs.
_required = ["describe_asset", "cmc_name_audit", "rural_reference", "mask_area_km2"]
_absent = [n for n in _required if not hasattr(aoi, n)]
if _absent:
    raise RuntimeError(
        f"colombo_uhi.aoi is missing {_absent} - it is STALE.\n"
        f"  loaded from: {aoi.__file__}\n"
        "Fix, in order:\n"
        "  1. Runtime > Restart session, then re-run from the CLONE cell.\n"
        "  2. If it persists, your local commits are not pushed - check that the\n"
        "     HEAD line printed by the clone cell is the revision you expect."
    )

for label, asset_key in (("DS (admin3)", "ds_divisions"), ("GN (admin4)", "gn_divisions")):
    asset_id = params["aoi"]["assets"][asset_key]
    if not asset_id:
        print(f"{label}: no asset configured (aoi.assets.{asset_key} is null)\n")
        continue
    info = aoi.describe_asset(asset_id, n_samples=2)
    print(f"{label}  ->  {info['asset_id']}")
    print(f"  features (nationwide): {info['count']}")
    print(f"  properties: {info['properties']}")
    for sample in info["samples"]:
        trimmed = {k: v for k, v in sample.items() if k != "system:index"}
        print(f"  sample: {trimmed}")
    print()

In [ ]:
# COLAB: RUN THIS CELL
# Administrative boundaries. DS/GN are filtered to Colombo District, so the
# counts must be 13 and 557 - NOT the nationwide 339 / 14043.
district = aoi.colombo_district(params)
province = aoi.western_province(params)
print("District features (expect 1):", district.size().getInfo())
print("Province features (expect 3):", province.size().getInfo())
print("Province districts:",
      province.aggregate_array(params["aoi"]["gaul"]["district_property"]).getInfo())

counts = params["aoi"]["expected_counts"]
ds_fc = aoi.ds_divisions(params)
gn_fc = aoi.gn_divisions(params)
for label, fc, expected_n in (("DS", ds_fc, counts["ds_divisions"]),
                              ("GN", gn_fc, counts["gn_divisions"])):
    got = fc.size().getInfo()
    verdict = "OK" if got == expected_n else f"<< CHECK (expected {expected_n})"
    print(f"\n{label} features in Colombo District: {got}  {verdict}")

ds_name_prop = params["aoi"]["assets"]["ds_name_property_candidates"][0]
print("DS names:", sorted(ds_fc.aggregate_array(ds_name_prop).getInfo()))

In [ ]:
# COLAB: RUN THIS CELL
# CMC = union of the 55 GN divisions the Colombo Municipal Council's own GIS Unit
# lists as inside the municipality. Audit the names FIRST: a partial match would
# silently produce an undersized CMC that still looks plausible, and duplicate GN
# names (they are NOT unique within the district) would silently enlarge it.
audit = aoi.cmc_name_audit(params)
print(f"definition   : {audit['definition']}  (name property '{audit['name_property']}')")
print(f"names matched: {len(audit['matched'])}/{audit['expected']}")
print(f"features     : {audit['matched_features']}  "
      f"({'OK' if audit['matched_features'] == audit['expected'] else '<< CHECK - duplicates?'})")
print(f"stated area  : {audit['stated_area_km2']:.2f} km2 (asset's own area field)")

if audit["missing"]:
    print(f"\nMISSING from the asset ({len(audit['missing'])}) - the CMC would be UNDERSIZED:")
    print("  ", audit["missing"])
    print("\nCandidate GN names NOT on our list - the asset's spelling of each")
    print("missing name should be in here:")
    print("  ", audit["extra"])
    print("\nFix: correct aoi.cmc.gn_division_names in config/params.yaml.")
else:
    print("\nAll configured GN names matched.")
    if audit["extra"]:
        print("Candidate GN divisions inside the parent DS divisions but NOT on the")
        print("CMC list (should be empty if the list is complete):")
        print("  ", audit["extra"])
    else:
        print("No leftover GN divisions - the CMC list exactly tiles its parent")
        print("DS divisions (Colombo + Thimbirigasyaya).")

cmc_geom = None
try:
    cmc_geom = aoi.cmc_boundary(params)
    print("\nCMC boundary built.")
except (RuntimeError, ValueError) as err:
    print("\nCMC boundary FAILED -", err)

In [ ]:
# COLAB: RUN THIS CELL
# Area sanity checks. GAUL is simplified at 500 m - a few % deviation is normal.
# The urban-extent vectorisation makes this cell take ~1 minute.
expected = params["aoi"]["expected_areas_km2"]

urban_geom = aoi.urban_extent(params)
ring_geom = aoi.buffer_ring(params) if cmc_geom is not None else None

rows = [
    ("Colombo District", district.geometry(10), expected["district"]),
    ("Western Province", province.geometry(10), expected["western_province"]),
    ("CMC (administrative)", cmc_geom, expected["cmc_administrative"]),
    ("Urban extent (GHSL)", urban_geom, None),
    ("Rural buffer ring", ring_geom, None),
]
print(f"{'AOI':<22}{'area km2':>12}{'expected':>12}")
for name, geom, exp in rows:
    if geom is None:
        print(f"{name:<22}{'- unavailable':>12}{exp or '':>12}")
        continue
    km2 = aoi.area_km2(geom).getInfo()
    flag = ""
    if exp:
        flag = "  OK" if abs(km2 - exp) / exp <= 0.10 else "  << CHECK (>10% off)"
    print(f"{name:<22}{km2:>12.1f}{exp or '':>12}{flag}")

# The CMC, three DIFFERENT numbers that must never be conflated:
#   (a) administrative - the raw polygon; legitimately exceeds the gazetted
#       37.31 km2 because COD-AB's Colombo DS encloses the Port outer harbour;
#   (b) land at 30 m   - polygon minus water on the analysis grid; the figure to
#       quote, and what LST statistics actually cover;
#   (c) land at 300 m  - the SAME quantity at the coarse scale used by the
#       mask-area table below. It differs by ~6% purely from raster
#       discretisation of a ragged coastal mask. Report the range, not one number
#       chosen for being closest to 37.31.
if cmc_geom is not None:
    gross = aoi.area_km2(cmc_geom).getInfo()
    land30 = aoi.cmc_land_area_km2(params).getInfo()                 # 30 m analysis grid
    land300 = aoi.cmc_land_area_km2(params, scale_m=300).getInfo()   # matches the table below
    gazetted = 37.31
    print(f"\nCMC administrative   : {gross:8.2f} km2  (asset-stated "
          f"{audit['stated_area_km2']:.2f}; = the Colombo + Thimbirigasyaya DS pair)")
    print(f"CMC land   @  30 m   : {land30:8.2f} km2  "
          f"({(land30 - gazetted) / gazetted:+.1%} vs gazetted {gazetted})")
    print(f"CMC land   @ 300 m   : {land300:8.2f} km2  "
          f"({(land300 - gazetted) / gazetted:+.1%} vs gazetted {gazetted})")
    print(f"water inside CMC     : {gross - land30:8.2f} km2  "
          "(Colombo Port harbour + Beira Lake + Kelani mouth)")
    print("\n-> quote the CMC land area WITH its scale; the 30-300 m spread is a"
          "\n   discretisation sensitivity, and the residual over 37.31 comes from"
          "\n   COD-AB polygon generalisation + the aoi.water_mask thresholds.")

In [ ]:
# COLAB: RUN THIS CELL
# Combined water mask: MNDWI OR QA_PIXEL-water-frequency OR JRC occurrence.
# Plus a 60 m shoreline-buffer variant (coastal mixed-pixel exclusion demo;
# the project default aoi.water_mask.shoreline_buffer_m is 0 = off).
water = aoi.water_mask(params)
water_buffered = aoi.water_exclusion_mask(params, shoreline_buffer_m=60)
print("Water mask built. Threshold config:", params["aoi"]["water_mask"])

In [ ]:
# COLAB: RUN THIS CELL
# BOTH SUHII rural-reference definitions behind the common interface.
# Later phases flip between them with this one string (CLAUDE.md caveat 5:
# always report the sensitivity, never a single number).
suhii = params["uhi"]["suhii"]
urban_lcz, rural_lcz = aoi.rural_reference("lcz_based", params)

# buffer_ring is based on the CMC, so it is skipped rather than allowed to kill
# the rest of the notebook when the DS name match failed above.
urban_br = rural_br = None
if cmc_geom is not None:
    urban_br, rural_br = aoi.rural_reference("buffer_ring", params)
else:
    print("SKIPPED buffer_ring rural reference - no CMC boundary.\n")

print("buffer_ring: ring", suhii["buffer_ring"]["inner_km"], "-",
      suhii["buffer_ring"]["outer_km"], "km beyond the",
      suhii["buffer_ring"]["base"], "; excludes", suhii["buffer_ring"]["exclude"])
print("lcz_based: urban =", suhii["lcz_based"]["urban_classes"],
      "| rural =", suhii["lcz_based"]["rural_classes"],
      "(A-G; water/class G removed by the water mask)")
print("lcz_based scope:", suhii["lcz_based"]["scope"],
      "- both LCZ masks are clipped to this geometry")
print("rural elevation cap:", suhii["rural_filters"]["max_elevation_m"], "m")
print()
print("The two definitions differ by design: buffer = CMC vs a 15-25 km ring;")
print("LCZ = built vs vegetated INSIDE the district. The LCZ rural reference sits")
print("closer to the core, so advection may damp its SUHII - report both.")

In [ ]:
# COLAB: RUN THIS CELL
# Mask areas - the guard against a rural reference the elevation cap has
# emptied. Reduced at 300 m for speed; these are approximate BY DESIGN, which is
# why the "urban - buffer_ring (CMC)" row here (~37.7) differs from the 30 m CMC
# land area printed above (~40.2). Same quantity, coarser grid - see that cell.
print(f"{'mask':<34}{'area km2':>12}")
for label, mask in (
    ("urban - buffer_ring (CMC)", urban_br),
    ("rural - buffer_ring", rural_br),
    ("urban - lcz_based (LCZ 1-10)", urban_lcz),
    ("rural - lcz_based (LCZ A-G)", rural_lcz),
    ("water mask", water),
):
    if mask is None:
        print(f"{label:<34}{'- skipped':>12}")
        continue
    km2 = aoi.mask_area_km2(mask, params, scale_m=300).getInfo()
    flag = "  << CHECK (near-empty)" if km2 < 5 else ""
    print(f"{label:<34}{km2:>12.1f}{flag}")

In [ ]:
# COLAB: RUN THIS CELL
# Static PNGs into figures/ - the interactive map below renders nothing once
# this notebook is saved, so these are the shareable verification evidence.
import ee
from IPython.display import Image, display

from colombo_uhi import viz

district_region = district.geometry(10).bounds(10)
province_region = aoi.analysis_region(params).bounds(10)
# Central Colombo, ~10 km around the city centre. The district-wide figure is
# ~50 m/px, too coarse to resolve Beira Lake (0.65 km2) or Diyawanna Lake; this
# zoom is ~22 m/px and does.
centre_pt = ee.Geometry.Point([params["aoi"]["centre"]["lon"],
                               params["aoi"]["centre"]["lat"]])
core_region = centre_pt.buffer(10000).bounds(10)
backdrop = viz.elevation_backdrop(params)

figures = {
    "aoi_boundaries.png": (
        province_region,
        [
            backdrop,
            viz.outline_image(province, "000000", 2),
            viz.outline_image(district, "d62728", 2),
            viz.outline_image(urban_geom, "ff7f0e", 2),
        ]
        + ([viz.outline_image(cmc_geom, "9467bd", 3)] if cmc_geom is not None else []),
    ),
    "aoi_water_mask.png": (
        district_region,
        [
            backdrop,
            water.selfMask().visualize(palette=["1f77b4"]),
            viz.outline_image(district, "d62728", 2),
        ],
    ),
    # Zoomed: Beira Lake, Diyawanna (Parliament) Lake, Kelani River mouth.
    "aoi_water_mask_core.png": (
        core_region,
        [
            backdrop,
            water.selfMask().visualize(palette=["1f77b4"]),
        ]
        + ([viz.outline_image(cmc_geom, "9467bd", 2)] if cmc_geom is not None else []),
    ),
    "aoi_rural_lcz.png": (
        district_region,
        [
            backdrop,
            rural_lcz.selfMask().visualize(palette=["2ca02c"]),
            urban_lcz.selfMask().visualize(palette=["d62728"]),
            viz.outline_image(district, "000000", 2),
        ],
    ),
}
if rural_br is not None:
    figures["aoi_rural_buffer_ring.png"] = (
        province_region,
        [
            backdrop,
            rural_br.selfMask().visualize(palette=["2ca02c"]),
            urban_br.selfMask().visualize(palette=["d62728"]),
            viz.outline_image(district, "000000", 1),
        ],
    )

for filename, (region, layers) in figures.items():
    path = viz.save_thumbnail(layers, region, os.path.join("figures", filename))
    print(path, "-", path.stat().st_size // 1024, "KB")
    display(Image(filename=str(path)))

In [ ]:
# COLAB: RUN THIS CELL
# Interactive map for zooming around. Toggle layers in the layer control.
# (Its output does NOT persist when the notebook is saved - see figures/ above.)
import geemap

centre = params["aoi"]["centre"]
m = geemap.Map(center=[centre["lat"], centre["lon"]], zoom=10)

m.addLayer(viz.outline_image(province, "000000"), {}, "Western Province (GAUL)")
m.addLayer(viz.outline_image(district, "d62728"), {}, "Colombo District (GAUL)")
m.addLayer(viz.outline_image(ds_fc, "8c564b", 1), {}, "DS divisions", False)
m.addLayer(viz.outline_image(gn_fc, "c49c94", 1), {}, "GN divisions", False)
if cmc_geom is not None:
    m.addLayer(viz.outline_image(cmc_geom, "9467bd", 3), {}, "CMC (from DS asset)")
m.addLayer(viz.outline_image(urban_geom, "ff7f0e"), {}, "Urban extent (GHSL)", False)
if ring_geom is not None:
    m.addLayer(viz.outline_image(ring_geom, "2ca02c"), {}, "Rural buffer ring (15-25 km)")

m.addLayer(water.selfMask(), {"palette": ["1f77b4"]}, "Water mask")
m.addLayer(water_buffered.selfMask(), {"palette": ["17becf"]},
           "Water mask + 60 m shoreline buffer", False)
if rural_br is not None:
    m.addLayer(rural_br.selfMask(), {"palette": ["98df8a"]},
               "Rural mask - buffer_ring method", False)
m.addLayer(urban_lcz.selfMask(), {"palette": ["7f7f7f"]},
           "Urban mask - LCZ classes 1-10", False)
m.addLayer(rural_lcz.selfMask(), {"palette": ["2ca02c"]},
           "Rural mask - LCZ A-G minus water", False)
m

## Phase 1 sign-off — VERIFIED in Colab run 5 (2026-08-08)

All values below were confirmed; they are the reference figures for the report.
Re-running this notebook should reproduce them.

| Quantity | Verified | Note |
|---|---|---|
| Colombo District | **685.6 km²**, 1 feature | GAUL, 500 m-simplified (gazetted 699) |
| Western Province | **3761.7 km²**, 3 features | Colombo + Gampaha + Kalutara |
| DS divisions in district | **13** | matches CLAUDE.md |
| GN divisions in district | **557** | matches CLAUDE.md |
| CMC name audit | **55/55 names, 55 features** | `missing` and `extra` both empty |
| CMC administrative | **47.07 km²** | = the Colombo + Thimbirigasyaya DS pair (46.87 stated) — the 55 GN divisions tile them exactly |
| CMC land @ 30 m | **40.18 km²** | +7.7% vs gazetted 37.31 |
| CMC land @ 300 m | **37.70 km²** | +1.0% — same quantity, coarser grid |
| water inside CMC | **6.89 km²** | Colombo Port harbour + Beira + Kelani mouth |
| Rural buffer ring (geometry) | **1603.5 km²** | 15–25 km annulus around the CMC |
| rural mask — buffer_ring | **206.1 km²** | after water, built-up and the 100 m cap |
| urban mask — buffer_ring | **37.7 km²** | water excluded from urban masks too |
| urban mask — LCZ 1–10 | **458.5 km²** | district-scoped |
| rural mask — LCZ A–G | **152.2 km²** | district-scoped, water + elevation applied |

**Figures** (all reviewed): `aoi_water_mask_core.png` confirms **Beira Lake**,
**Diyawanna (Parliament) Lake** and the **Kelani River**;
`aoi_water_mask.png` confirms the ocean, **Bolgoda Lake** and the
Labugama/Kalatuwawa reservoirs; `aoi_boundaries.png` shows one compact CMC nested
inside district inside province; `aoi_rural_buffer_ring.png` shows a single
coastal core with a landward-only ring that stops short of the high ground.

### Carry these caveats into later phases

1. **CMC area is scale-dependent** — always quote the reduction scale with it.
   The residual over the gazetted 37.31 km² is COD-AB polygon generalisation plus
   the `aoi.water_mask` thresholds; report it as sensitivity, do not tune it away.
2. **GN names are not unique within Colombo District.** Any GN-level filter must
   be scoped to its parent DS division (or use `adm4_pcode`). Relevant to the
   GN-level zonal statistics and MAUP work in Phases 5–7.
3. The two rural definitions differ **by design** (CMC-vs-ring against
   built-vs-vegetated inside the district), so their SUHII values will differ.
   Reporting both is the requirement, not a problem to resolve.

**Next:** Phase 2 — `02_lst_pipeline.ipynb` (harmonised Landsat C2 L2 + MODIS LST).

## Appendix - re-uploading the DS/GN boundary assets

GAUL stops at district level for Sri Lanka, so DS divisions, GN divisions and
the CMC boundary need **user-uploaded EE assets** (see PROGRESS.md, 2026-08-08).

1. Download **"Sri Lanka - Subnational Administrative Boundaries"** from
   OCHA/HDX (<https://data.humdata.org/dataset/cod-ab-lka>): the **admin3**
   shapefile = DS divisions, **admin4** = GN divisions. Check the licence before
   redistributing derived maps.
2. In the EE Code Editor (<https://code.earthengine.google.com>):
   *Assets > New > Shape files* - upload each shapefile set
   (`.shp .shx .dbf .prj` together).
3. Put the asset ids into `config/params.yaml` under `aoi.assets.ds_divisions`
   / `aoi.assets.gn_divisions`, commit, push, re-run.
4. The uploaded layers cover **all of Sri Lanka**; `aoi.ds_divisions()` and
   `aoi.gn_divisions()` filter to Colombo District for you - by attribute when
   the asset has a parent-district column, otherwise by a
   centroid-within-district spatial test (geoBoundaries has no such column).